<a href="https://colab.research.google.com/github/pksheaad/AI_Learning/blob/main/Lab/Lab_1_summarize_dialogue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative AI Use Case: Summarize Dialogue

In this lab, you will do the dialogue summarization task using generative AI. You will explore how the input text affects the output of the model, and perform prompt engineering to direct it towards the task you need. By comparing zero shot, one shot, and few shot inferences, you will take the first step towards prompt engineering and see how it can enhance the generative output of Large Language Models.

# Table of Contents

- [ 1 - Set up Required Dependencies](#1)
- [ 2 - Summarize Dialogue without Prompt Engineering](#2)
- [ 3 - Summarize Dialogue with an Instruction Prompt](#3)
  - [ 3.1 - Zero Shot Inference with an Instruction Prompt](#3.1)
  - [ 3.2 - Zero Shot Inference with the Prompt Template from FLAN-T5](#3.2)
- [ 4 - Summarize Dialogue with One Shot and Few Shot Inference](#4)
  - [ 4.1 - One Shot Inference](#4.1)
  - [ 4.2 - Few Shot Inference](#4.2)
- [ 5 - Generative Configuration Parameters for Inference](#5)


1 - Set up Required Dependencies

In [ ]:
# !pip install -U \
#     torch==2.5.1 \
#     datasets==2.17.0 \
#     transformers==4.38.2

In [ ]:
!pip install transformers==4.38.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 34.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the 

In [1]:
import warnings
warnings.filterwarnings("ignore")

# import dependencies
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    GenerationConfig
)

<a name='2'></a>
## 2 - Summarize Dialogue without Prompt Engineering

In this use case, you will be generating a summary of a dialogue with the pre-trained Large Language Model (LLM) FLAN-T5 from Hugging Face. The list of available models in the Hugging Face `transformers` package can be found [here](https://huggingface.co/docs/transformers/index).

Let's upload some simple dialogues from the [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) Hugging Face dataset. This dataset contains 10,000+ dialogues with the corresponding manually labeled summaries and topics.

In [2]:
huggingface_dataset_name = "knkarthick/dialogsum"
dataset = load_dataset(path = huggingface_dataset_name)
dataset

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [3]:
# Print a couple of dialogues with their baseline summaries.
example_indeces = [40,100]
for i, index in enumerate(example_indeces,1):
  print("-"*100)
  print(f"Example: {i}")
  print("-"*100)
  print("INPUT DIALOGUE")
  print(dataset['test'][index]['dialogue'])
  print("-"*100)
  print("BASE LINE HUMAN SUMMARY")
  print(dataset['test'][index]['summary'])
  print("-"*100)
  print()


----------------------------------------------------------------------------------------------------
Example: 1
----------------------------------------------------------------------------------------------------
INPUT DIALOGUE
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
----------------------------------------------------------------------------------------------------
BASE LINE HUMAN SUMMARY
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------

Load the [FLAN-T5 model](https://huggingface.co/docs/transformers/model_doc/flan-t5), creating an instance of the `AutoModelForSeq2SeqLM` class with the `.from_pretrained()` method.

In [4]:
model_name = "google/flan-t5-base"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
#model

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

To perform encoding and decoding, you need to work with text in a tokenized form. **Tokenization** is the process of splitting texts into smaller units that can be processed by the LLM models.

Download the tokenizer for the FLAN-T5 model using `AutoTokenizer.from_pretrained()` method. Parameter `use_fast` switches on fast tokenizer. At this stage, there is no need to go into the details of that, but you can find the tokenizer parameters in the [documentation](https://huggingface.co/docs/transformers/v4.28.1/en/model_doc/auto#transformers.AutoTokenizer).

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = True)
#tokenizer

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [6]:
# Test the tokenizer encoding and decoding a simple sentence:
sentence = "What time is it, Tom?"
sentence_encoded = tokenizer(sentence, return_tensors = "pt")
sentence_encoded
print(f"Encoded Sentence: {sentence_encoded['input_ids'][0]}")

Encoded Sentence: tensor([ 363,   97,   19,   34,    6, 3059,   58,    1])


In [7]:
# decoding
decoded_sentence = tokenizer.decode(token_ids = sentence_encoded.get('input_ids')[0], skip_special_tokens = True)
decoded_sentence
print(f"Decoded Sentence: {decoded_sentence}")

Decoded Sentence: What time is it, Tom?


Now it's time to explore how well the base LLM summarizes a dialogue without any prompt engineering. **Prompt engineering** is an act of a human changing the **prompt** (input) to improve the response for a given task.

In [8]:
for i, index in enumerate(example_indeces, 1):
  dialogue = dataset['test'][index]['dialogue']
  summary = dataset['test'][index]['summary']
  inputs = tokenizer(dialogue, return_tensors = "pt")
  outputs = tokenizer.decode(
      model.generate(
          inputs['input_ids'],
          max_new_tokens = 50
      )[0],
      skip_special_tokens = True
  )
  print("-"*100)
  print(f"Example {i}")
  print("-"*100)
  print(f"INPUT PROMPT\n: {dialogue}")
  print("-"*100)
  print(f"SUMMARY: {summary}")
  print("-"*100)
  print(f"MODEL RESPONSE: {outputs}")
  print("-"*100)
  print()







----------------------------------------------------------------------------------------------------
Example 1
----------------------------------------------------------------------------------------------------
INPUT PROMPT
: #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
----------------------------------------------------------------------------------------------------
SUMMARY: #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
----------------------------------------------------------------------------------------------------
MODEL RESPONSE: Person1: It's ten to nine.
--------------------------------------------------------------------------

You can see that the guesses of the model make some sense, but it doesn't seem to be sure what task it is supposed to accomplish. Seems it just makes up the next sentence in the dialogue. Prompt engineering can help here.

## 3 - Summarize Dialogue with an Instruction Prompt

Prompt engineering is an important concept in using foundation models for text generation. You can check out [this blog](https://www.amazon.science/blog/emnlp-prompt-engineering-is-the-new-feature-engineering) from Amazon Science for a quick introduction to prompt engineering.

<a name='3.1'></a>
### 3.1 - Zero Shot Inference with an Instruction Prompt

In order to instruct the model to perform a task - summarize a dialogue - you can take the dialogue and convert it into an instruction prompt. This is often called **zero shot inference**.  You can check out [this blog from AWS](https://aws.amazon.com/blogs/machine-learning/zero-shot-prompting-for-the-flan-t5-foundation-model-in-amazon-sagemaker-jumpstart/) for a quick description of what zero shot learning is and why it is an important concept to the LLM model.

Wrap the dialogue in a descriptive instruction and see how the generated text will change:

In [9]:
for i, index in enumerate(example_indeces):
  dialogue = dataset['test'][index]['dialogue']
  summary = dataset['test'][index]['summary']

  prompt = f"""
  Summarized the following converstation based on given dialogue

  Dialogue: {dialogue}

  Summary:
  """
  inputs = tokenizer(prompt, return_tensors = "pt")
  outputs = tokenizer.decode(
      model.generate(
          inputs['input_ids'],
          max_new_tokens = 50
      )[0],
      skip_special_tokens = True
  )
  print("-"*100)
  print(f"Example {i}")
  print("-"*100)
  print(f"INPUT PROMPT\n: {dialogue}")
  print("-"*100)
  print(f"SUMMARY: {summary}")
  print("-"*100)
  print(f"MODEL RESPONSE- ZERO SHOT: {outputs}")
  print("-"*100)
  print()


----------------------------------------------------------------------------------------------------
Example 0
----------------------------------------------------------------------------------------------------
INPUT PROMPT
: #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
----------------------------------------------------------------------------------------------------
SUMMARY: #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
----------------------------------------------------------------------------------------------------
MODEL RESPONSE- ZERO SHOT: Tom is late. He has to catch the nine-thirty train.
--------------------------------------

**Exercise:**

- Experiment with the `prompt` text and see how the inferences will be changed. Will the inferences change if you end the prompt with just empty string vs. `Summary: `?
- Try to rephrase the beginning of the `prompt` text from `Summarize the following conversation.` to something different - and see how it will influence the generated output.

In [10]:
for i, index in enumerate(example_indeces):
  dialogue = dataset['test'][index]['dialogue']
  summary = dataset['test'][index]['summary']

  prompt = f"""
  Provide the summary of conversation

  Dialogue: {dialogue}

  Summary:
  """
  inputs = tokenizer(prompt, return_tensors = "pt")
  outputs = tokenizer.decode(
      model.generate(
          inputs['input_ids'],
          max_new_tokens = 50
      )[0],
      skip_special_tokens = True
  )
  print("-"*100)
  print(f"Example {i}")
  print("-"*100)
  print(f"INPUT PROMPT\n: {dialogue}")
  print("-"*100)
  print(f"SUMMARY: {summary}")
  print("-"*100)
  print(f"MODEL RESPONSE- ZERO SHOT: {outputs}")
  print("-"*100)
  print()


----------------------------------------------------------------------------------------------------
Example 0
----------------------------------------------------------------------------------------------------
INPUT PROMPT
: #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
----------------------------------------------------------------------------------------------------
SUMMARY: #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
----------------------------------------------------------------------------------------------------
MODEL RESPONSE- ZERO SHOT: Tom is late for the train.
---------------------------------------------------------------

<a name='3.2'></a>
### 3.2 - Zero Shot Inference with the Prompt Template from FLAN-T5

Let's use a slightly different prompt. FLAN-T5 has many prompt templates that are published for certain tasks [here](https://github.com/google-research/FLAN/tree/main/flan/v2). In the following code, you will use one of the [pre-built FLAN-T5 prompts](https://github.com/google-research/FLAN/blob/main/flan/v2/templates.py):

In [11]:
for i, index in enumerate(example_indeces):
  dialogue = dataset['test'][index]['dialogue']
  summary = dataset['test'][index]['summary']

  prompt = f"""
  Provide the summary of conversation

  Dialogue: {dialogue}

  What was going on?
  """
  inputs = tokenizer(prompt, return_tensors = "pt")
  outputs = tokenizer.decode(
      model.generate(
          inputs['input_ids'],
          max_new_tokens = 50
      )[0],
      skip_special_tokens = True
  )
  print("-"*100)
  print(f"Example {i}")
  print("-"*100)
  print(f"INPUT PROMPT\n: {dialogue}")
  print("-"*100)
  print(f"SUMMARY: {summary}")
  print("-"*100)
  print(f"MODEL RESPONSE- ZERO SHOT: {outputs}")
  print("-"*100)
  print()

----------------------------------------------------------------------------------------------------
Example 0
----------------------------------------------------------------------------------------------------
INPUT PROMPT
: #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
----------------------------------------------------------------------------------------------------
SUMMARY: #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
----------------------------------------------------------------------------------------------------
MODEL RESPONSE- ZERO SHOT: The train is about to leave Tom's place.
-------------------------------------------------

<a name='4'></a>
## 4 - Summarize Dialogue with One Shot and Few Shot Inference

**One shot and few shot inference** are the practices of providing an LLM with either one or more full examples of prompt-response pairs that match your task - before your actual prompt that you want completed. This is called "in-context learning" and puts your model into a state that understands your specific task.  You can read more about it in [this blog from HuggingFace](https://huggingface.co/blog/few-shot-learning-gpt-neo-and-inference-api).

<a name='4.1'></a>
### 4.1 - One Shot Inference

Let's build a function that takes a list of `example_indices_full`, generates a prompt with full examples, then at the end appends the prompt which you want the model to complete (`example_index_to_summarize`).  You will use the same FLAN-T5 prompt template from section [3.2](#3.2).

In [31]:
from typing import List
def make_prompt(example_indices_full:List[int], example_index_to_summarize:int)->str:
  prompt = ""
  for index in example_indices_full:
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    prompt += f"""
    Dialogue:
    {dialogue}

    What was going on?:
    {summary}
    """

    dialogue = dataset['test'][example_index_to_summarize]['dialogue']

    prompt = prompt + f"""
    Dialogue:
    {dialogue}

    What was going on?
    """
  return prompt

In [33]:
# test the function
example_indices_full = [40]
example_index_to_summarize = 200

print(make_prompt(example_indices_full, example_index_to_summarize))


    Dialogue: 
  #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

  What was going on?:
  #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
    
    Dialogue:
    #Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin w

Now pass this prompt to perform the one shot inference:

In [41]:
# Get the one shot prompt
one_shot_prompt = make_prompt(example_indices_full = example_indices_full, example_index_to_summarize = example_index_to_summarize)
summary = dataset['test'][example_index_to_summarize]['summary']

input = tokenizer(one_shot_prompt, return_tensors = "pt")
output = tokenizer.decode(
  model.generate(input['input_ids'], max_new_tokens = 50)[0],
  skip_special_tokens = True
)
print("-"*100)
print(f"BASE HUMAN SUMMARY:\n {summary}")
print(f"MODEL ONE SHOT RESPONSE:\n {output}")

----------------------------------------------------------------------------------------------------
BASE HUMAN SUMMARY:
 #Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
MODEL ONE SHOT RESPONSE:
 #Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to add a CD-ROM drive.


<a name='4.2'></a>
### 4.2 - Few Shot Inference

Let's explore few shot inference by adding two more full dialogue-summary pairs to your prompt.

In [45]:
example_indices_full = [40, 80, 120]
prompt = make_prompt(example_indices_full = example_indeces, example_index_to_summarize = example_index_to_summarize)
print(prompt)


    Dialogue: 
  #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

  What was going on?:
  #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
    
    Dialogue:
    #Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin w

In [48]:
example_indices_full = [40, 80, 120]
summary = dataset['test'][example_index_to_summarize]['summary']
few_more_shot = make_prompt(example_indices_full = example_indeces, example_index_to_summarize = example_index_to_summarize)
few_more_shot_tokens = tokenizer(few_more_shot, return_tensors = "pt")
output = tokenizer.decode(
    model.generate(
        few_more_shot_tokens['input_ids'],
        max_new_tokens = 50
    )[0],
    skip_special_tokens = True
)

print(f"HUMAN SUMMARY:\n {summary}")
print(f"FEW MORE SHOT MODEL RESPONSE:\n {output}")


HUMAN SUMMARY:
 #Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
FEW MORE SHOT MODEL RESPONSE:
 #Person1 and #Person2 are considering upgrading their systems.


<a name='5'></a>
## 5 - Generative Configuration Parameters for Inference

You can change the configuration parameters of the `generate()` method to see a different output from the LLM. So far the only parameter that you have been setting was `max_new_tokens=50`, which defines the maximum number of tokens to generate. A full list of available parameters can be found in the [Hugging Face Generation documentation](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationConfig).

A convenient way of organizing the configuration parameters is to use `GenerationConfig` class.

In [55]:
#generation_config = GenerationConfig(max_new_tokens = 50)
#generation_config = GenerationConfig(max_new_tokens = 50, do_sample = True, temperature = 0.1)
#generation_config = GenerationConfig(max_new_tokens = 50, do_sample = True, temperature = 0.5)
generation_config = GenerationConfig(max_new_tokens = 50, do_sample = True, temperature = 1)

response = tokenizer.decode(
    model.generate(few_more_shot_tokens['input_ids'],
                   generation_config = generation_config)[0],
    skip_special_tokens = True
)

print("-"*70)
print(f"Model Resposne:\n {response}")
print("-"*70)

----------------------------------------------------------------------
Model Resposne:
 #Person1 proposed adding a painting program to #Person2's software. #Person1 proposes adding a DVD drive. All software programs running on CDs are on CDs.
----------------------------------------------------------------------


## 5 - Complete example with FLAN-T5

In [56]:
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    GenerationConfig
)

In [57]:
model_name = "google/flan-t5-small"

In [66]:
# instantiate the model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# tokenizer for encoding
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [62]:
dialogue = """
John: Are you coming to the office tomorrow?
Mary: No, I will be working from home.
John: Okay, I will inform the manager.
"""

In [63]:
prompt = f""" Create summary of the converstaion based on given dialogue.
Dialogue:
{dialogue}
Summary:
"""

In [71]:
# generation config
generation_config = GenerationConfig(new_max_tokens = 100, do_sample = True, temperature = 0.1)


In [72]:
input_tokens = tokenizer(prompt, return_tensors = "pt")
response = tokenizer.decode(
    model.generate(input_tokens['input_ids'],
                   generation_config = generation_config)[0],
    skip_special_tokens = True
)

In [73]:
print(f"MODEL RESPOSNE:\n {response}")

MODEL RESPOSNE:
 Mary will be working from home. John will inform the manager.
